In [1]:
import requests
import torch
import time
import psutil
import subprocess

import pandas as pd

from datasets import load_dataset

In [18]:
ds = load_dataset("KushT/bbc_news_multiclass_train_val_test")

val = ds['validation'].to_pandas()

val.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 379 entries, 0 to 378
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    379 non-null    object
 1   label   379 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 6.1+ KB


In [ ]:
val['label'] = val['label'].apply(lambda x: 'business' if x == 0 else 'entertainment' if x == 1 else 'politics' if x == 2 else 'sport' if x == 3 else 'tech')

labels = val['label'].unique()

val

In [3]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_13792\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


In [4]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [13]:
def generate_text(message, url):
    payload = {
        "model": "deepseek-r1:1.5b",
        "prompt": message,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "num_predict": 1000,
        },
        "raw": True
    }

    response = requests.post(url, json=payload)

    return response.json()

In [11]:
def classify(text, labels):

    url = "http://localhost:11434/api/generate"

    message = f"""[INST]
        System: You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions.
        User: Classify the following text based on the task: Category classification of news articles. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}
        [/INST]
        """

    start_time = time.time()
    response = generate_text(message, url)
    response_time = time.time() - start_time

    done_reason = response['done_reason']
    num_tokens = response['eval_count']
    response = response['response'].strip().lower()


    if done_reason != 'length':
        content = response.split('</think>')[1]
    else:
        content = "loop"

    vram_usage = get_gpu_memory_usage()

    ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)

    if 'business' in content:
        content = 'business'
    elif 'sport' in content:
        content = 'sport'
    elif 'entertainment' in content:
        content = 'entertainment'
    elif 'politics' in content:
        content = 'politics'
    elif 'tech' in content:
        content = 'tech'
    else:
        content = 'error'

    print(content, num_tokens)

    return content, num_tokens

In [19]:
val = val.sample(frac=0.1, random_state=42)

In [20]:
# columns 'prediction' and 'num_tokens' are added to the dataframe through apply
val['prediction'], val['num_tokens'] = zip(*val['text'].apply(lambda x: classify(x, labels)))

C:\Users\Rafael\AppData\Local\Temp\ipykernel_13792\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


sport 761
error 1000
politics 448
entertainment 341
tech 522
tech 18
business 346
sport 375
error 203
tech 436
business 590
business 913
sport 741
sport 493
politics 33
business 649
error 310
error 1000
tech 16
tech 374
sport 312
business 494
tech 449
tech 500
entertainment 806
tech 875
tech 79
politics 782
error 1000
error 51
business 347
entertainment 520
business 370
sport 351
business 213
business 303
entertainment 839
error 1000


In [21]:
val

,text,label,prediction,num_tokens
288,Klinsmann issues Lehmann warning Germany coach...,3,sport,761
283,Celebrities get to stay in jungle All four con...,1,error,1000
327,UK 'needs true immigration data' A former Home...,2,politics,448
145,Rap boss arrested over drug find Rap mogul Mar...,1,entertainment,341
55,Fockers retain film chart crown Comedy Meet Th...,1,tech,522
93,"Gamer buys $26,500 virtual land A 22-year-old ...",4,tech,18
341,WorldCom bosses' $54m payout Ten former direct...,0,business,346
82,IAAF will contest Greek decision The Internati...,3,sport,375
366,Bombardier chief to leave company Shares in tr...,0,error,203
148,Green reports shun supply chain Nearly 20% mor...,0,tech,436


In [22]:
# in val dataset, calculate average of all num_tokens that are less than 1000. if they are 1000 or bigger, dont consider them on the average
val['num_tokens'] = val['num_tokens'].apply(lambda x: x if x < 1000 else None)
average_num_tokens = val['num_tokens'].mean()

In [23]:
average_num_tokens

np.float64(437.05882352941177)